# 🚀 .NET 8 Web API on Google Colab

This notebook:
1. Installs .NET 8 SDK
2. Uploads and runs your Web API
3. Exposes it publicly via **ngrok**

**Endpoints available after start:**
| Method | Path | Description |
|--------|------|-------------|
| GET | `/` | Welcome message |
| GET | `/health` | Health check |
| GET | `/greet/{name}` | Personalised greeting |
| GET | `/todos` | List all todos |
| POST | `/todos` | Create a todo |
| PUT | `/todos/{id}` | Update a todo |
| DELETE | `/todos/{id}` | Delete a todo |
| GET | `/calc?a=5&b=3&op=add` | Calculator (add/subtract/multiply/divide) |
| GET | `/swagger` | Swagger UI |

## Step 1 — Install .NET 8 SDK

In [ ]:
%%bash
# Install .NET 8
wget -q https://packages.microsoft.com/config/ubuntu/22.04/packages-microsoft-prod.deb -O packages-microsoft-prod.deb
dpkg -i packages-microsoft-prod.deb
apt-get update -q
apt-get install -y dotnet-sdk-8.0 > /dev/null 2>&1
dotnet --version

## Step 2 — Write the API source files

In [ ]:
import os
os.makedirs('/content/ColabApi', exist_ok=True)

In [ ]:
%%writefile /content/ColabApi/ColabApi.csproj
<Project Sdk="Microsoft.NET.Sdk.Web">
  <PropertyGroup>
    <TargetFramework>net8.0</TargetFramework>
    <Nullable>enable</Nullable>
    <ImplicitUsings>enable</ImplicitUsings>
    <RootNamespace>ColabApi</RootNamespace>
  </PropertyGroup>
  <ItemGroup>
    <PackageReference Include="Swashbuckle.AspNetCore" Version="6.5.0" />
  </ItemGroup>
</Project>

In [ ]:
%%writefile /content/ColabApi/Program.cs
var builder = WebApplication.CreateBuilder(args);

builder.Services.AddEndpointsApiExplorer();
builder.Services.AddSwaggerGen();

builder.Services.AddCors(options =>
    options.AddPolicy("AllowAll", p =>
        p.AllowAnyOrigin().AllowAnyMethod().AllowAnyHeader()));

var app = builder.Build();
app.UseSwagger();
app.UseSwaggerUI();
app.UseCors("AllowAll");

app.MapGet("/", () => new {
    message = "Hello from .NET on Google Colab!",
    version = "1.0",
    time    = DateTime.UtcNow
});

app.MapGet("/health", () => new {
    status = "healthy",
    uptime = Environment.TickCount64 / 1000,
    time   = DateTime.UtcNow
});

app.MapGet("/greet/{name}", (string name) => new {
    message = $"Hello, {name}!",
    time    = DateTime.UtcNow
});

var todos = new List<Todo>
{
    new(1, "Buy groceries", false),
    new(2, "Read a book",   true),
    new(3, "Go for a walk", false),
};

app.MapGet("/todos", () => todos);

app.MapGet("/todos/{id:int}", (int id) =>
{
    var todo = todos.FirstOrDefault(t => t.Id == id);
    return todo is not null ? Results.Ok(todo) : Results.NotFound(new { error = $"Todo {id} not found" });
});

app.MapPost("/todos", (TodoRequest req) =>
{
    var todo = new Todo(todos.Max(t => t.Id) + 1, req.Title, false);
    todos.Add(todo);
    return Results.Created($"/todos/{todo.Id}", todo);
});

app.MapPut("/todos/{id:int}", (int id, TodoRequest req) =>
{
    var todo = todos.FirstOrDefault(t => t.Id == id);
    if (todo is null) return Results.NotFound(new { error = $"Todo {id} not found" });
    todos.Remove(todo);
    var updated = todo with { Title = req.Title, Done = req.Done };
    todos.Add(updated);
    return Results.Ok(updated);
});

app.MapDelete("/todos/{id:int}", (int id) =>
{
    var todo = todos.FirstOrDefault(t => t.Id == id);
    if (todo is null) return Results.NotFound(new { error = $"Todo {id} not found" });
    todos.Remove(todo);
    return Results.Ok(new { message = $"Todo {id} deleted" });
});

app.MapGet("/calc", (double a, double b, string op) =>
{
    var result = op.ToLower() switch
    {
        "add"      => a + b,
        "subtract" => a - b,
        "multiply" => a * b,
        "divide"   => b != 0 ? a / b : double.NaN,
        _          => double.NaN
    };
    return new { a, b, op, result };
});

app.Run("http://0.0.0.0:5000");

record Todo(int Id, string Title, bool Done);
record TodoRequest(string Title, bool Done = false);

## Step 3 — Build the project

In [ ]:
!dotnet build /content/ColabApi/ColabApi.csproj -c Release

## Step 4 — Start the API in the background

In [ ]:
import subprocess, time, os

log_file = open('/content/api.log', 'w')
proc = subprocess.Popen(
    ['dotnet', 'run', '--project', '/content/ColabApi/ColabApi.csproj',
     '--no-build', '-c', 'Release'],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print('⏳ Waiting for API to start...')
time.sleep(8)

# Quick smoke-test
import urllib.request, json
try:
    with urllib.request.urlopen('http://localhost:5000/health') as r:
        data = json.loads(r.read())
        print(f'✅ API is up! Status: {data["status"]}')
except Exception as e:
    print(f'❌ API not responding yet: {e}')
    print('Check /content/api.log for errors')

## Step 5 — Expose publicly with ngrok

> **Get a free token at https://ngrok.com** → Dashboard → Your Authtoken

In [ ]:
!pip install pyngrok -q

In [ ]:
from pyngrok import ngrok

# 🔑 Paste your ngrok token here
NGROK_TOKEN = 'PASTE_YOUR_TOKEN_HERE'

ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(5000)
base = str(public_url).strip()

print('=' * 55)
print(f'🌐  Base URL    : {base}')
print(f'📋  Swagger UI  : {base}/swagger')
print(f'❤️   Health      : {base}/health')
print(f'📝  Todos       : {base}/todos')
print(f'👋  Greet       : {base}/greet/YourName')
print(f'🔢  Calculator  : {base}/calc?a=10&b=3&op=multiply')
print('=' * 55)

## Step 6 — Test the API right here in Colab

In [ ]:
import urllib.request, json

def call(method, path, body=None):
    url = f'http://localhost:5000{path}'
    data = json.dumps(body).encode() if body else None
    headers = {'Content-Type': 'application/json'} if data else {}
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    with urllib.request.urlopen(req) as r:
        return json.loads(r.read())

print('GET /           :', call('GET', '/'))
print('GET /health     :', call('GET', '/health'))
print('GET /greet/Alice:', call('GET', '/greet/Alice'))
print('GET /todos      :', call('GET', '/todos'))
print('POST /todos     :', call('POST', '/todos', {'title': 'New task'}))
print('GET /calc       :', call('GET', '/calc?a=10&b=4&op=multiply'))

## Step 7 — View logs (optional)

In [ ]:
!tail -30 /content/api.log

## Step 8 — Stop the API

In [ ]:
ngrok.kill()
proc.terminate()
print('✅ API and tunnel stopped.')